In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

In [ ]:
REPO_ID = "local/mbdroid_franka_pick"
ROOT    = Path("./lerobot_out")
ds = LeRobotDataset(REPO_ID, root=ROOT)
print(f"episodes={ds.num_episodes}  frames={ds.num_frames}  fps={ds.fps}")

In [ ]:
import pandas as pd
info = json.loads((ROOT / "meta/info.json").read_text())
rows = [{"feature": k, "dtype": v["dtype"], "shape": tuple(v["shape"])} for k, v in info["features"].items()]
df = pd.DataFrame(rows)
df

In [ ]:
s = ds[0]
state = s["observation.state"].numpy()
action = s["action"].numpy()
print(f"observation.state  shape={state.shape}")
print(f"  eef_9d   [0:9]  = {np.round(state[:9], 3)}")
print(f"  gripper  [9:10] = {np.round(state[9:10], 3)}")
print(f"  joints   [10:17]= {np.round(state[10:17], 3)}")
print(f"\naction             shape={action.shape}")
print(f"  eef_9d   [0:9]  = {np.round(action[:9], 3)}")
print(f"  gripper  [9:10] = {np.round(action[9:10], 3)}")
print(f"  joints   [10:17]= {np.round(action[10:17], 3)}")
print(f"\ntask: {s['task']}")

cam_keys = [k for k in s if k.startswith("observation.images.")]
fig, axes = plt.subplots(1, len(cam_keys), figsize=(4 * len(cam_keys), 3))
if len(cam_keys) == 1:
    axes = [axes]
for ax, k in zip(axes, cam_keys):
    img = s[k].numpy()
    if img.ndim == 3 and img.shape[0] in (1, 3):  # CHW -> HWC
        img = img.transpose(1, 2, 0)
    img = img.astype(np.float32)
    
    if img.max() <= 1.0:
        img = (img * 255).clip(0, 255).astype(np.uint8)
    else:
        img = img.astype(np.uint8)
    ax.imshow(img)
    ax.set_title(k.rsplit(".", 1)[-1])
    ax.axis("off")
fig.suptitle(f"episode {int(s['episode_index'])}  frame {int(s['frame_index'])}")
plt.tight_layout()